# 02 — Model A Calibration: Reliability Diagrams, Brier Scores, and Conformal Coverage

This notebook evaluates the **probability calibration** and **uncertainty quantification** of Model A.

### Why Calibration is Critical in NYAYA:
The decision engine calculates expected harm in rupee currency units:
$$H_F = P(\text{fraud}) \times \text{amount}$$
$$H_H = (1 - P(\text{fraud})) \times \beta e^{\alpha S} u$$
$$H_\Delta = H_F - H_H$$

If $P(\text{fraud})$ is uncalibrated, expected harm comparisons are distorted, leading to either wrongful fund withholding for starving families or uninhibited syndicate looting.

### Key Objectives:
1. **Load Saved Metrics**: Inspect reference values from `artifacts/metrics_a.json`, `calibration.json`, and `conformal.json`.
2. **Recompute Calibration**: Fit/apply Isotonic Regression on held-out calibration data and compute calibrated probabilities $p_{\text{cal}}$.
3. **Brier Score Evaluation**: Quantify calibration error before and after isotonic regression ($Brier = \frac{1}{N}\sum (p_i - y_i)^2$).
4. **Reliability Diagrams**: Generate equal-width reliability diagrams comparing predicted vs observed fraud rates.
5. **Conformal Coverage**: Evaluate split conformal prediction coverage ($1 - \alpha = 90\%$) and analyze prediction sets across populations.

In [ ]:
import sys
import os
from pathlib import Path

# Dataset / model tag
TAG = "v2"

# Add project root to sys.path
cwd = Path.cwd()
for candidate in [cwd, cwd.parent, Path("/home/ibm-user/shared/nyaya-kaustub")]:
    if (candidate / "models").is_dir() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))
        break

# Resolve paths
DATA_DIR = Path(os.environ.get("NYAYA_DATA", "/home/ibm-user/shared/data"))
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd().parent / "data"

ARTIFACTS_DIR = Path(os.environ.get("NYAYA_ARTIFACTS", "/home/ibm-user/shared/nyaya/artifacts"))
if not ARTIFACTS_DIR.exists():
    ARTIFACTS_DIR = Path.cwd().parent / "artifacts"

print(f"Dataset / Model Tag:      {TAG}")
print(f"Project root on sys.path: {sys.path[0]}")
print(f"Data directory:           {DATA_DIR} (exists: {DATA_DIR.exists()})")
print(f"Artifacts directory:      {ARTIFACTS_DIR} (exists: {ARTIFACTS_DIR.exists()})")

In [ ]:
import json
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, roc_auc_score

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except ImportError:
    MATPLOTLIB_AVAILABLE = False
    print("Notice: matplotlib not present in environment. Outputting tabular summaries.")

from data.features import MODEL_A_FEATURES, DISASTER_CONTEXT, tx_features_frame
from models.common import SEED, brier, reliability, split_indices
from models.train_a import FRACTIONS, PARAMS, ALPHA, conformal_qhat, conformal_sets

print(f"Calibration Configuration: alpha={ALPHA} (target coverage={1-ALPHA:.1%})")

## 1. Load Saved Values from Artifacts
Load reference metrics from `artifacts/metrics_a.json`, `calibration.json`, and `conformal.json` when available.

In [ ]:
metrics_file = ARTIFACTS_DIR / "metrics_a.json"
cal_file = ARTIFACTS_DIR / "calibration.json"
conf_file = ARTIFACTS_DIR / "conformal.json"

saved_metrics = None
if metrics_file.exists():
    saved_metrics = json.loads(metrics_file.read_text())
    print(f"--- Saved Pipeline Metrics ({metrics_file.name}) ---")
    print(f"Version:              {saved_metrics.get('version')}")
    print(f"Best Iteration:       {saved_metrics.get('best_iteration')}")
    print(f"Test PR-AUC:          {saved_metrics['test'].get('pr_auc'):.4f}")
    print(f"Test ROC-AUC:         {saved_metrics['test'].get('roc_auc'):.4f}")
    print(f"Test Brier (Raw):     {saved_metrics['test'].get('brier_raw'):.5f}")
    print(f"Test Brier (Cal):     {saved_metrics['test'].get('brier_calibrated'):.5f}")
    print(f"Conformal qhat:       {saved_metrics['test']['conformal'].get('qhat'):.4f}")
    print(f"Conformal Coverage:   {saved_metrics['test']['conformal'].get('coverage'):.3%}")
    print(f"Conformal Share Both: {saved_metrics['test']['conformal'].get('share_both'):.3%}")
else:
    print(f"Artifact {metrics_file} not found; proceeding to compute directly from model.")

## 2. Load Dataset and Feature Matrix
Load transactions from `nyaya_v2.csv.gz` and build the Model A feature matrix.

In [ ]:
data_file = DATA_DIR / f"nyaya_{TAG}.csv.gz"
print(f"Loading {data_file}...")
df = pd.read_csv(data_file)
X = tx_features_frame(df)
y = df["is_fraud"].to_numpy(dtype=int)

strata = df["population"].astype(str) + "|" + df["is_fraud"].astype(str)
sp = split_indices(strata, FRACTIONS, seed=SEED)
d = {k: (X.iloc[v], y[v]) for k, v in sp.items()}

print(f"Partitions: train={len(sp['train']):,}, val={len(sp['val']):,}, iso={len(sp['iso']):,}, conf={len(sp['conf']):,}, test={len(sp['test']):,}")

## 3. Obtain Predictions Across Splits
Load `model_a.json` booster (or train on the train split if missing) and generate raw prediction scores on the calibration split (`iso`) and held-out test split (`test`).

In [ ]:
model_file = ARTIFACTS_DIR / "model_a.json"
if model_file.exists():
    print(f"Loading trained booster from {model_file}...")
    booster = xgb.Booster()
    booster.load_model(str(model_file))
else:
    print("Training booster on train split...")
    dtrain = xgb.DMatrix(d["train"][0], d["train"][1], feature_names=MODEL_A_FEATURES)
    dval = xgb.DMatrix(d["val"][0], d["val"][1], feature_names=MODEL_A_FEATURES)
    booster = xgb.train(PARAMS, dtrain, num_boost_round=600, evals=[(dval, "val")],
                        early_stopping_rounds=30, verbose_eval=False)

def predict_raw(features_df):
    dm = xgb.DMatrix(features_df, feature_names=MODEL_A_FEATURES)
    return booster.predict(dm)

raw_iso = predict_raw(d["iso"][0])
y_iso = d["iso"][1]

Xt, yt = d["test"]
p_raw_test = predict_raw(Xt)

print(f"Raw scores generated: iso split ({len(raw_iso):,}), test split ({len(p_raw_test):,})")

## 4. Recompute Isotonic Calibration and Brier Score
Apply Isotonic Regression to map raw scores into calibrated probabilities:
$$p_{\text{cal}} = \text{interp}(p_{\text{raw}}, X_{\text{thresholds}}, y_{\text{thresholds}})$$
Compare Brier scores before and after calibration against saved metrics.

In [ ]:
if cal_file.exists():
    print(f"Loading calibration thresholds from {cal_file}...")
    cal_data = json.loads(cal_file.read_text())
    iso_x = np.asarray(cal_data["x"], float)
    iso_y = np.asarray(cal_data["y"], float)
else:
    print("Fitting IsotonicRegression on iso split...")
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0).fit(raw_iso, y_iso)
    iso_x, iso_y = iso.X_thresholds_, iso.y_thresholds_

# Calibrated probabilities on held-out test split
p_cal_test = np.interp(p_raw_test, iso_x, iso_y)

recomputed_brier_raw = brier(yt, p_raw_test)
recomputed_brier_cal = brier(yt, p_cal_test)
brier_improvement = (recomputed_brier_raw - recomputed_brier_cal) / recomputed_brier_raw

print("--- Brier Score Evaluation (Recomputed) ---")
print(f"Raw Brier Score:        {recomputed_brier_raw:.5f}")
print(f"Calibrated Brier Score: {recomputed_brier_cal:.5f}")
print(f"Brier Error Reduction:  {brier_improvement:.2%}")

if saved_metrics:
    saved_raw = saved_metrics["test"]["brier_raw"]
    saved_cal = saved_metrics["test"]["brier_calibrated"]
    print(f"\nConsistency Verification with Saved Artifact:")
    print(f"  Raw Brier:        Saved={saved_raw:.5f} | Recomputed={recomputed_brier_raw:.5f} | Delta={abs(saved_raw - recomputed_brier_raw):.2e}")
    print(f"  Calibrated Brier: Saved={saved_cal:.5f} | Recomputed={recomputed_brier_cal:.5f} | Delta={abs(saved_cal - recomputed_brier_cal):.2e}")

## 5. Reliability Diagrams Before and After Calibration
Calculate empirical reliability across 10 equal-width bins using `models.common.reliability`.
A perfectly calibrated model tracks the diagonal $y = x$ line, meaning that among transactions assigned probability $p \approx 0.4$, exactly 40% are actual fraud.

In [ ]:
rel_raw = reliability(yt, p_raw_test, bins=10)
rel_cal = reliability(yt, p_cal_test, bins=10)

rel_raw_df = pd.DataFrame(rel_raw).rename(columns={"mean_pred": "raw_mean_p", "observed": "raw_observed"})
rel_cal_df = pd.DataFrame(rel_cal).rename(columns={"mean_pred": "cal_mean_p", "observed": "cal_observed"})
merged_rel = pd.merge(rel_raw_df, rel_cal_df, on=["bin", "n"], how="outer").sort_values("bin").reset_index(drop=True)

print("--- Reliability Table: Raw vs Calibrated ---")
print(merged_rel.to_string(index=False))

if MATPLOTLIB_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Before calibration
    axes[0].plot([0, 1], [0, 1], "k--", label="Ideal (y = x)")
    if len(rel_raw) > 0:
        axes[0].plot([r["mean_pred"] for r in rel_raw], [r["observed"] for r in rel_raw], "o-", color="#e06666", label="Raw XGBoost")
    axes[0].set_xlabel("Mean Predicted Probability")
    axes[0].set_ylabel("Observed Fraud Fraction")
    axes[0].set_title(f"Before Calibration (Brier: {recomputed_brier_raw:.5f})")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    # After calibration
    axes[1].plot([0, 1], [0, 1], "k--", label="Ideal (y = x)")
    if len(rel_cal) > 0:
        axes[1].plot([r["mean_pred"] for r in rel_cal], [r["observed"] for r in rel_cal], "s-", color="#4a86e8", label="Isotonic Calibrated")
    axes[1].set_xlabel("Mean Predicted Probability")
    axes[1].set_ylabel("Observed Fraud Fraction")
    axes[1].set_title(f"After Calibration (Brier: {recomputed_brier_cal:.5f})")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

## 6. Conformal Prediction Coverage
Evaluate split conformal prediction:
- Non-conformity score: $s_i = 1 - p(\text{true label})$.
- Calibrated quantile: $\hat{q} = \text{Quantile}(s, \lceil(n+1)(1-\alpha)\rceil / n)$.
- Conformal set: $\{\text{legit} \text{ if } p \le \hat{q}\} \cup \{\text{fraud} \text{ if } 1 - p \le \hat{q}\}$.
- Validates that marginal coverage meets the mathematical guarantee of $1 - \alpha = 90\%$.

In [ ]:
# Compute qhat on conformal split
raw_conf = predict_raw(d["conf"][0])
p_cal_conf = np.interp(raw_conf, iso_x, iso_y)
y_conf = d["conf"][1]

recomputed_qhat = conformal_qhat(p_cal_conf, y_conf, alpha=ALPHA)

if conf_file.exists():
    saved_qhat = json.loads(conf_file.read_text())["qhat"]
    print(f"Conformal Quantile: Saved qhat = {saved_qhat:.4f} | Recomputed qhat = {recomputed_qhat:.4f}")
    qhat = saved_qhat
else:
    qhat = recomputed_qhat
    print(f"Recomputed qhat = {qhat:.4f}")

# Conformal sets on test split
test_sets = conformal_sets(p_cal_test, qhat)
covered = np.where(
    yt == 1,
    (test_sets == "fraud") | (test_sets == "both"),
    (test_sets == "legit") | (test_sets == "both")
)
empirical_coverage = float(covered.mean())

print(f"--- Conformal Coverage Verification (Target: {1 - ALPHA:.1%}) ---")
print(f"Empirical Coverage on Test Split: {empirical_coverage:.2%}")
print(f"Singleton 'legit':                 {(test_sets == 'legit').mean():.2%}")
print(f"Singleton 'fraud':                 {(test_sets == 'fraud').mean():.2%}")
print(f"Uncertain Set {'{legit, fraud}'}:       {(test_sets == 'both').mean():.2%}")
print(f"Empty Set:                         {(test_sets == 'empty').mean():.2%}")

# Conformal set composition across populations
pop_test = df["population"].iloc[sp["test"]].to_numpy()
conf_pop_rows = []
for p in sorted(set(pop_test)):
    m = pop_test == p
    conf_pop_rows.append({
        "population": p,
        "n": int(m.sum()),
        "coverage": float(covered[m].mean()),
        "share_legit": float((test_sets[m] == "legit").mean()),
        "share_fraud": float((test_sets[m] == "fraud").mean()),
        "share_both": float((test_sets[m] == "both").mean()),
        "share_empty": float((test_sets[m] == "empty").mean())
    })

conf_pop_df = pd.DataFrame(conf_pop_rows)
print("\n--- Conformal Set Distribution Across Populations ---")
print(conf_pop_df.to_string(index=False))